In [1]:
#CONEXION AL ORACLE
import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()




In [2]:
base2 = pd.read_sql_query(f"SELECT * FROM mbtoe10", con=connection2)
a=base2[base2.duplicated(subset=["topemecod"])]
print(a)
base2.to_sql(name='tmp_mbtoe10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL mbtoe10 FALSO CON LO OBTENIDO DEL ORACLE

tabla='mbtoe10'
query_count_before = f"SELECT COUNT(*) FROM {tabla}"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")


query="""
ALTER TABLE tmp_mbtoe10 
ALTER COLUMN topemecod TYPE CHARACTER (2),
ALTER COLUMN topemedes TYPE CHARACTER (25),
ALTER COLUMN topemetriflg TYPE CHARACTER (1);


UPDATE mbtoe10 
SET 

topemedes= t.topemedes,
topemetriflg= t.topemetriflg


FROM tmp_mbtoe10 t 
WHERE mbtoe10.topemecod = t.topemecod and mbtoe10.topemecod IS NOT NULL and t.topemecod IS NOT NULL ;

INSERT INTO mbtoe10 
(topemecod, topemedes, topemetriflg) 

SELECT 
topemecod, topemedes, topemetriflg

FROM tmp_mbtoe10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM mbtoe10 
    WHERE mbtoe10.topemecod = tmp_mbtoe10.topemecod and mbtoe10.topemecod IS NOT NULL and tmp_mbtoe10.topemecod IS NOT NULL
);


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_mbtoe10;
DELETE FROM mbtoe10 WHERE topemecod ='';
"""


c2= text(query2)
cursor=connection3.execute(c2)


query_count_after = f"SELECT COUNT(*) FROM {tabla}"
cant_despues = connection3.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")


base2 = pd.read_sql_query(f"SELECT * FROM mbtoe10", con=connection3)



Empty DataFrame
Columns: [topemecod, topemedes, topemetriflg]
Index: []
Cantidad de filas en la tabla mbtoe10 antes de la inserción: 351
Cantidad de filas en la tabla mbtoe10 después de la inserción: 354
La cantidad de filas insertadas fue de: 3


In [3]:


base2.to_sql(name='tmp_mbtoe10', con=engine4, if_exists='replace', index=False)


tabla='dim_emetop'
query_count_before = f"SELECT COUNT(*) FROM {tabla}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")


query="""

ALTER TABLE tmp_mbtoe10 
ALTER COLUMN topemecod TYPE CHARACTER (2),
ALTER COLUMN topemedes TYPE CHARACTER (25),
ALTER COLUMN topemetriflg TYPE CHARACTER (1);


INSERT INTO dim_emetop (cod_top, des_top,fla_top) 
SELECT topemecod, topemedes,topemetriflg

FROM tmp_mbtoe10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_emetop 
    WHERE dim_emetop.cod_top = tmp_mbtoe10.topemecod
);

DROP TABLE tmp_mbtoe10;
"""

c1 = text(query)
cursor = connection4.execute(c1)

query_count_after = f"SELECT COUNT(*) FROM {tabla}"
cant_despues = connection4.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")



Cantidad de filas en la tabla dim_emetop antes de la inserción: 351
Cantidad de filas en la tabla dim_emetop después de la inserción: 354
La cantidad de filas insertadas fue de: 3


In [4]:
connection4.close()
connection3.close()
connection2.close()
